In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import re
from scipy import stats
import mysql.connector

In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="avito_db"
)

cursor = conn.cursor()

In [ ]:
query = "SELECT * FROM terrain_Transformed"
df = pd.read_sql(query, conn)
df.head()

<h2 style="color:black"> Binary attributes Vizualisation </h2>

In [ ]:
attrs = ['Loti', 'Titre']

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))

for i, feat in enumerate(attrs):
    sns.countplot(x=feat, data=df, ax=axes[i], width=0.4, color='orange')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Count')

plt.tight_layout()
plt.show()


<h2 style="color:black"> Numerical attributes Vizualisation </h2>

In [ ]:
features2 = ['Surface_Totale', 'Quartier_enc', 'Zoning_enc']

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 6))

for i, feat in enumerate(features2):
    ax = axes[i // 2, i % 2]
    
    if feat == "Surface Totale":
        sns.histplot(df[feat], binwidth=50, color='orange', ax=ax)
        ax.set_xlim(0, 20000)
    elif feat == "Quartier_enc":
        sns.histplot(df[feat], binwidth=10000, color='orange', ax=ax)
        ax.set_xlim(0, 5500000)
    else:
        sns.histplot(df[feat], binwidth=1, color='orange', ax=ax)
        ax.set_xlim(0, 6)
    
    ax.set_xlabel(feat)
    ax.set_ylabel('Frequency')

# Supprimer le 4e subplot vide si seulement 3 features
if len(features2) < 4:
    fig.delaxes(axes[1, 1])

plt.tight_layout()
plt.show()


In [ ]:
features2 = ['Prix', 'Surface_Totale', 'Quartier_enc', 'Zoning_enc']
new_df = df[features2].copy()

In [ ]:
correlation_matrix = new_df.corr()

plt.figure(figsize=(11,11))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix for numeric features in the dataset')
plt.xticks(fontsize=6)
plt.yticks(fontsize=6)
plt.show()

In [ ]:
features1 = ['Prix', 'Loti', 'Titre']
new_df_1 = df[features1].copy()

In [ ]:
# We use the Pearson Correlation method for the binary attributes
corr_matrix = new_df_1.corr(method='pearson')
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f',xticklabels=new_df_1.columns, yticklabels=new_df_1.columns)
plt.title('Correlation Matrix of Binary Features')
plt.show()

In [ ]:
df = pd.get_dummies(df, columns=['Ville'])
df.columns

In [ ]:
villes = ['Ville_Autres',
       'Ville_Casablanca et alentours', 'Ville_Fès-Meknès', 'Ville_Kénitra',
       'Ville_Marrakech', 'Ville_Rabat et alentours',
       'Ville_Tanger et alentours']
for ville in villes:
    df[ville] = df[ville].map({True: 1, False: 0})

df.head()

<h2 style="color:black"> Features selection:</h2>

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


X = df.drop(columns=['Prix', 'Quartier', 'Zoning', 'Lien']) 

# Standardize the features by scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


pca = PCA(n_components=5)

# Fit and transform the scaled data
X_pca = pca.fit_transform(X_scaled)

# Create a DataFrame with PCA components
df_pca = pd.DataFrame(data=X_pca, columns=['PCA_Component_1', 'PCA_Component_2', 'PCA_Component_3','PCA_Component_4','PCA_Component_5'])

# Concatenate the PCA components with the target variable (if applicable)

df_final = pd.concat([df_pca, df['Prix']], axis=1)
# Use df_final for further analysis or visualization

# Explained variance ratio
explained_var_ratio = pca.explained_variance_ratio_
print(f"Explained variance ratio: {explained_var_ratio}")

# If you want to visualize the PCA components (2 components for example)
plt.scatter(df_pca['PCA_Component_1'], df_pca['PCA_Component_3'], color="#0080ff")
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('PCA Components')
plt.show()

In [ ]:
import numpy as np

# Remplacer les vrais NaN (np.nan) ou chaînes "nan"/"NaN" par None
df = df.replace(["nan", "NaN", np.nan], None)

# Liste des colonnes sans 'id'
columns = [col for col in df.columns if col != "id"]

# Construction de la requête
cols_str = ", ".join(f"`{col}`" for col in columns)
placeholders = ", ".join(["%s"] * len(columns))
insert_query = f"INSERT INTO terrain_Transformed_Explored ({cols_str}) VALUES ({placeholders})"

# Insertion dans MySQL
for _, row in df[columns].iterrows():
    cursor.execute(insert_query, tuple(row))

conn.commit()


In [ ]:
cursor.close()
conn.close()
